# Qwen3-1.7B Sinhala QA fine-tuning on Modal CUDA

LoRA fine-tuning for context-grounded Sinhala history QA using `new_split_v2/train.jsonl`. This is the CUDA adaptation of Hugging Face's Qwen3 Optimum Neuron tutorial; the final test split remains untouched.

In [ ]:
# Run this only in a fresh Modal kernel. Do not replace Modal's Torch/NumPy wheels.
# Transformers 4.51.3 supports Qwen3 without importing Torch Flex Attention.
%uv pip install --reinstall --no-deps "transformers==4.51.3" "peft==0.15.2" "accelerate==1.7.0" "tokenizers==0.21.4" "huggingface_hub==0.35.3" "safetensors==0.6.2" "hf_xet==1.1.10"
print("Packages installed. Restart this fresh kernel once before running the next cell.")

In [ ]:
import json
import math
import os
import random
import re
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
import transformers

if transformers.__version__ != "4.51.3":
    raise RuntimeError(
        f"Transformers {transformers.__version__} is loaded instead of 4.51.3. "
        "Rerun the install cell with --reinstall, then restart the kernel."
    )

# Modal's optional torchvision/sklearn packages are not needed for text QA.
# Mark them unavailable before PEFT triggers Transformers' lazy model imports.
from transformers.utils import import_utils as transformers_import_utils

transformers_import_utils._torchvision_available = False
transformers_import_utils._sklearn_available = False
assert not transformers_import_utils.is_torchvision_available()
assert not transformers_import_utils.is_sklearn_available()

import accelerate
import peft
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

EXPECTED_VERSIONS = {
    "transformers": "4.51.3",
    "peft": "0.15.2",
    "accelerate": "1.7.0",
}
actual_versions = {
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "accelerate": accelerate.__version__,
}
if actual_versions != EXPECTED_VERSIONS:
    raise RuntimeError(
        f"Package versions are {actual_versions}, expected {EXPECTED_VERSIONS}. "
        "Restart the Modal kernel and rerun from the configuration cell."
    )
print("Package versions:", actual_versions)
print("Transformers location:", transformers.__file__)
print("PEFT location:", peft.__file__)
print("NumPy from Modal image:", np.__version__)
print("Optional vision/sklearn imports disabled for this text-only run")
print("Torch:", torch.__version__)
print("Torch location:", torch.__file__)

MODEL_ID = "Qwen/Qwen3-1.7B"
OUTPUT_DIR = Path("qwen3_1_7b_sinhala_qa_stable_v4")
ADAPTER_DIR = OUTPUT_DIR / "adapter"
MERGED_DIR = OUTPUT_DIR / "merged"

SEED = 42
MAX_LENGTH = 1536
VALIDATION_RATIO = 0.10
UNANSWERABLE_TRAIN_FRACTION = 0.20
MAX_NEW_TOKENS = 256
SANITY_EXAMPLES_PER_CLASS = 3
# Evaluate the LoRA adapter first. Merge only after its outputs are verified.
MERGE_MODEL = False
PUSH_TO_HUB = False
HF_REPO_ID = "your-hf-username/qwen3-1.7b-sinhala-qa"

TRAIN_FILE_CANDIDATES = [
    Path("/tmp/train.jsonl"),
    Path("/tmp/new_split_v2/train.jsonl"),
    Path("new_split_v2/train.jsonl"),
    Path("../new_split_v2/train.jsonl"),
]
TRAIN_FILE = next((path for path in TRAIN_FILE_CANDIDATES if path.is_file()), None)
if TRAIN_FILE is None:
    checked = "\n".join(f"  - {path}" for path in TRAIN_FILE_CANDIDATES)
    raise FileNotFoundError(f"Could not find train.jsonl. Checked:\n{checked}")

if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA GPU in Modal before running this notebook.")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

model_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("GPU:", torch.cuda.get_device_name(0))
print("Model:", MODEL_ID)
print("Dtype:", model_dtype)
print("Train file:", TRAIN_FILE.resolve())

In [ ]:
REQUIRED_FIELDS = {
    "answer",
    "answerable",
    "chapter",
    "chapter_title",
    "context",
    "grade",
    "question",
}


def normalize_text(value):
    value = unicodedata.normalize("NFC", value)
    return re.sub(r"\s+", " ", value).strip()


records = []
with TRAIN_FILE.open("r", encoding="utf-8-sig") as handle:
    for line_number, line in enumerate(handle, 1):
        if not line.strip():
            continue
        item = json.loads(line)
        missing = REQUIRED_FIELDS.difference(item)
        if missing:
            raise ValueError(f"Line {line_number} is missing fields: {sorted(missing)}")
        if type(item["answerable"]) is not bool:
            raise TypeError(f"Line {line_number}: answerable must be a JSON boolean")
        for field in ("context", "question", "answer"):
            if not isinstance(item[field], str):
                raise TypeError(f"Line {line_number}: {field} must be a string")
        if not item["context"].strip() or not item["question"].strip():
            raise ValueError(f"Line {line_number}: context and question cannot be empty")
        if item["answerable"] and not item["answer"].strip():
            raise ValueError(f"Line {line_number}: answerable example has an empty answer")
        records.append(item)

if not records:
    raise ValueError("The training file contains no examples")

print(f"Loaded {len(records)} training records from {TRAIN_FILE.resolve()}")

group_to_indexes = defaultdict(list)
for index, item in enumerate(records):
    group_to_indexes[normalize_text(item["context"])].append(index)

group_keys = list(group_to_indexes)
validation_group_count = max(1, math.ceil(len(group_keys) * VALIDATION_RATIO))
source_false_ratio = sum(not item["answerable"] for item in records) / len(records)
best_split = None
best_score = float("inf")

for attempt in range(100):
    shuffled_groups = group_keys.copy()
    random.Random(SEED + attempt).shuffle(shuffled_groups)
    validation_groups = set(shuffled_groups[:validation_group_count])
    validation_indexes = [
        index
        for group in validation_groups
        for index in group_to_indexes[group]
    ]
    train_indexes = [
        index
        for group in shuffled_groups[validation_group_count:]
        for index in group_to_indexes[group]
    ]

    train_labels = {records[index]["answerable"] for index in train_indexes}
    validation_labels = {
        records[index]["answerable"] for index in validation_indexes
    }
    if train_labels != {True, False} or validation_labels != {True, False}:
        continue

    validation_false_ratio = (
        sum(not records[index]["answerable"] for index in validation_indexes)
        / len(validation_indexes)
    )
    validation_size_ratio = len(validation_indexes) / len(records)
    score = abs(validation_false_ratio - source_false_ratio) + abs(
        validation_size_ratio - VALIDATION_RATIO
    )
    if score < best_score:
        best_score = score
        best_split = (train_indexes, validation_indexes)

if best_split is None:
    raise RuntimeError("Could not create a grouped validation split with both labels")

train_indexes, validation_indexes = best_split
train_rows = [records[index] for index in train_indexes]
validation_rows = [records[index] for index in validation_indexes]

train_contexts = {normalize_text(item["context"]) for item in train_rows}
validation_contexts = {normalize_text(item["context"]) for item in validation_rows}
assert train_contexts.isdisjoint(validation_contexts)
assert {item["answerable"] for item in train_rows} == {True, False}
assert {item["answerable"] for item in validation_rows} == {True, False}


def oversample_unanswerable(rows, target_fraction, seed):
    if not 0 < target_fraction < 0.5:
        raise ValueError("UNANSWERABLE_TRAIN_FRACTION must be between 0 and 0.5")
    answerable_rows = [item for item in rows if item["answerable"]]
    unanswerable_rows = [item for item in rows if not item["answerable"]]
    target_unanswerable_count = math.ceil(
        target_fraction * len(answerable_rows) / (1 - target_fraction)
    )
    extra_count = max(0, target_unanswerable_count - len(unanswerable_rows))
    extra_rows = [
        unanswerable_rows[index % len(unanswerable_rows)]
        for index in range(extra_count)
    ]
    balanced_rows = list(rows) + extra_rows
    random.Random(seed).shuffle(balanced_rows)
    return balanced_rows


trainer_train_rows = oversample_unanswerable(
    train_rows,
    UNANSWERABLE_TRAIN_FRACTION,
    SEED,
)


def print_counts(name, rows):
    labels = Counter(item["answerable"] for item in rows)
    print(
        f"{name}: total={len(rows)}, "
        f"answerable={labels[True]}, unanswerable={labels[False]}"
    )


print_counts("All training split", records)
print_counts("Trainer train before oversampling", train_rows)
print_counts("Trainer train after oversampling", trainer_train_rows)
print_counts("Trainer validation", validation_rows)
print("Shared train/validation contexts: 0")

In [ ]:
NO_ANSWER = "සපයා ඇති සන්දර්භයේ පිළිතුර නොමැත."

SYSTEM_PROMPT = """You are a helpful Sinhala history question-answering assistant.

Answer the question using ONLY information explicitly provided in the context.

Instructions:
- Read the entire context before answering.
- Use no external knowledge, assumptions, or guesses.
- Match the person or entity named in the question exactly.
- Use evidence that contains both the requested entity and requested attribute.
- Do not take facts from nearby sentences about a different entity or event.
- If the answer is not explicitly supported by the context, respond exactly with: "සපයා ඇති සන්දර්භයේ පිළිතුර නොමැත."
- Return only the final answer in natural Sinhala.
- Do not explain your reasoning or mention source references."""

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if not tokenizer.is_fast:
    raise RuntimeError("A fast tokenizer is required for safe context-window offsets")
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


def make_user_message(item, context=None):
    if context is None:
        context = item["context"]
    return (
        f"Context:\n\n{context.strip()}\n\n"
        f"Question:\n\n{item['question'].strip()}"
    )


def find_subsequence(sequence, subsequence):
    if not subsequence or len(subsequence) > len(sequence):
        return -1
    last_start = len(sequence) - len(subsequence)
    for start in range(last_start + 1):
        if sequence[start : start + len(subsequence)] == subsequence:
            return start
    return -1


def select_context_window(item, answer, max_context_tokens):
    context_text = item["context"].strip()
    context_encoding = tokenizer(
        context_text,
        add_special_tokens=False,
        return_offsets_mapping=True,
    )
    context_ids = context_encoding["input_ids"]
    offsets = context_encoding["offset_mapping"]
    if len(context_ids) <= max_context_tokens:
        return context_text

    window_start = None
    if item["answerable"]:
        answer_start = -1
        answer_token_count = 0
        answer_char_start = context_text.find(answer)
        if answer_char_start >= 0:
            answer_char_end = answer_char_start + len(answer)
            answer_token_indexes = [
                index
                for index, (start, end) in enumerate(offsets)
                if end > answer_char_start and start < answer_char_end
            ]
            if answer_token_indexes:
                answer_start = answer_token_indexes[0]
                answer_token_count = len(answer_token_indexes)

        if answer_start < 0:
            answer_ids = tokenizer(answer, add_special_tokens=False)["input_ids"]
            answer_start = find_subsequence(context_ids, answer_ids)
            answer_token_count = len(answer_ids)

        if answer_start >= 0:
            room_before = max(0, (max_context_tokens - answer_token_count) // 2)
            window_start = max(0, answer_start - room_before)
            window_start = min(
                window_start,
                len(context_ids) - max_context_tokens,
            )

    if window_start is None:
        relevance_text = item["question"]
        if item["answerable"]:
            relevance_text += "\n" + answer
        relevance_ids = set(
            tokenizer(relevance_text, add_special_tokens=False)["input_ids"]
        )
        stride = max(1, max_context_tokens // 4)
        candidate_starts = list(
            range(0, len(context_ids) - max_context_tokens + 1, stride)
        )
        final_start = len(context_ids) - max_context_tokens
        if candidate_starts[-1] != final_start:
            candidate_starts.append(final_start)
        window_start = max(
            candidate_starts,
            key=lambda start: sum(
                token_id in relevance_ids
                for token_id in context_ids[start : start + max_context_tokens]
            ),
        )

    window_ids = context_ids[window_start : window_start + max_context_tokens]
    return tokenizer.decode(window_ids, skip_special_tokens=True).strip()


def build_token_ids(item, answer, context):
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_message(item, context)},
    ]
    full_messages = prompt_messages + [
        {"role": "assistant", "content": answer},
    ]

    prompt_ids = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    full_ids = tokenizer.apply_chat_template(
        full_messages,
        tokenize=True,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    return prompt_ids, full_ids


def tokenize_record(item):
    answer = item["answer"].strip() if item["answerable"] else NO_ANSWER
    prompt_ids, full_ids = build_token_ids(item, answer, item["context"])
    was_truncated = False

    if len(full_ids) > MAX_LENGTH:
        _, empty_context_ids = build_token_ids(item, answer, "")
        context_budget = MAX_LENGTH - len(empty_context_ids) - 8
        if context_budget < 32:
            raise ValueError("MAX_LENGTH is too small for the question and answer")

        while context_budget >= 32:
            cropped_context = select_context_window(item, answer, context_budget)
            prompt_ids, full_ids = build_token_ids(item, answer, cropped_context)
            if len(full_ids) <= MAX_LENGTH:
                was_truncated = True
                break
            context_budget -= len(full_ids) - MAX_LENGTH + 8

    if full_ids[: len(prompt_ids)] != prompt_ids:
        raise ValueError("The assistant response is not aligned with the prompt prefix")

    if len(full_ids) > MAX_LENGTH:
        raise ValueError(
            f"Example still requires {len(full_ids)} tokens after context cropping"
        )

    encoded = {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": [-100] * len(prompt_ids) + full_ids[len(prompt_ids) :],
    }
    supervised_labels = [label for label in encoded["labels"] if label != -100]
    if not supervised_labels:
        raise ValueError("Example has no supervised assistant tokens")
    if tokenizer.eos_token_id not in supervised_labels:
        raise ValueError("Assistant target does not contain the EOS token")
    return encoded, was_truncated


class TokenizedQADataset(torch.utils.data.Dataset):
    def __init__(self, rows):
        self.items = []
        self.truncated_count = 0
        for item in rows:
            encoded, was_truncated = tokenize_record(item)
            self.items.append(encoded)
            self.truncated_count += int(was_truncated)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        return self.items[index]


print("Tokenizing training examples...")
train_tokenized = TokenizedQADataset(trainer_train_rows)
print("Tokenizing validation examples...")
validation_tokenized = TokenizedQADataset(validation_rows)
train_crop_ratio = train_tokenized.truncated_count / len(train_tokenized)
validation_crop_ratio = validation_tokenized.truncated_count / len(validation_tokenized)
print(
    "Cropped training contexts:",
    f"{train_tokenized.truncated_count}/{len(train_tokenized)} "
    f"({train_crop_ratio:.1%})",
)
print(
    "Cropped validation contexts:",
    f"{validation_tokenized.truncated_count}/{len(validation_tokenized)} "
    f"({validation_crop_ratio:.1%})",
)
if train_tokenized.truncated_count or validation_tokenized.truncated_count:
    print(
        "Warning: long contexts were cropped to fit MAX_LENGTH. "
        "Answerable windows retain the gold answer; unanswerable windows use "
        "question-token relevance."
    )

all_lengths = [len(item["input_ids"]) for item in train_tokenized]
all_lengths += [len(item["input_ids"]) for item in validation_tokenized]
print("Maximum tokenized length:", max(all_lengths))
print("Mean tokenized length:", round(sum(all_lengths) / len(all_lengths), 2))
print("All examples fit MAX_LENGTH:", max(all_lengths) <= MAX_LENGTH)
print("Loss is masked over prompt tokens and applied only to assistant answers.")

for expected_answerable in (True, False):
    sample_index = next(
        index
        for index, item in enumerate(trainer_train_rows)
        if item["answerable"] is expected_answerable
    )
    sample = train_tokenized[sample_index]
    sample_target_ids = [
        token_id
        for token_id, label in zip(sample["input_ids"], sample["labels"])
        if label != -100
    ]
    print(
        f"Supervised target check (answerable={expected_answerable}):",
        tokenizer.decode(sample_target_ids).strip(),
    )

In [ ]:
print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=model_dtype,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
model.config.use_cache = False

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
unexpected_trainable = [
    name
    for name, parameter in model.named_parameters()
    if parameter.requires_grad and ("embed_tokens" in name or "lm_head" in name)
]
if unexpected_trainable:
    raise RuntimeError(f"Embedding/output parameters unexpectedly trainable: {unexpected_trainable}")
model.enable_input_require_grads()
model.print_trainable_parameters()

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
    return_tensors="pt",
)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    overwrite_output_dir=True,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.10,
    weight_decay=0.01,
    max_grad_norm=1.0,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=model_dtype == torch.bfloat16,
    fp16=model_dtype == torch.float16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=validation_tokenized,
    data_collator=data_collator,
)

print("Trainer ready.")

In [ ]:
train_result = trainer.train(resume_from_checkpoint=False)
print(train_result)
if not math.isfinite(train_result.training_loss):
    raise RuntimeError("Training loss is not finite; do not save this adapter")

evaluation = trainer.evaluate()
print("Validation metrics:", evaluation)
if not math.isfinite(evaluation["eval_loss"]):
    raise RuntimeError("Validation loss is not finite; do not save this adapter")
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Run the adapter sanity-check cells before saving or merging.")

In [ ]:
# Always validate the adapter itself before any optional merge.
inference_model = trainer.model
inference_model.config.use_cache = True
inference_model.generation_config.do_sample = False
inference_model.generation_config.temperature = None
inference_model.generation_config.top_p = None
inference_model.generation_config.top_k = None
print("Inference model: trained LoRA adapter (not merged)")

In [ ]:
def make_inference_inputs(context, question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"Context:\n\n{context.strip()}\n\nQuestion:\n\n{question.strip()}",
        },
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=True,
        return_tensors="pt",
    )


def prepare_inference_inputs(context, question, max_new_tokens):
    inputs = make_inference_inputs(context, question)
    prompt_limit = MAX_LENGTH - max_new_tokens
    was_cropped = False

    if inputs["input_ids"].shape[-1] > prompt_limit:
        empty_inputs = make_inference_inputs("", question)
        context_budget = prompt_limit - empty_inputs["input_ids"].shape[-1] - 8
        if context_budget < 32:
            raise ValueError("MAX_LENGTH is too small for this inference question")

        inference_item = {
            "context": context,
            "question": question,
            "answerable": False,
        }
        while context_budget >= 32:
            cropped_context = select_context_window(
                inference_item,
                "",
                context_budget,
            )
            inputs = make_inference_inputs(cropped_context, question)
            overflow = inputs["input_ids"].shape[-1] - prompt_limit
            if overflow <= 0:
                was_cropped = True
                break
            context_budget -= overflow + 8

    if inputs["input_ids"].shape[-1] > prompt_limit:
        raise ValueError("Inference prompt still exceeds MAX_LENGTH after cropping")
    return inputs.to(inference_model.device), was_cropped


def ask_from_context(
    context,
    question,
    max_new_tokens=MAX_NEW_TOKENS,
    debug=False,
):
    inputs, was_cropped = prepare_inference_inputs(
        context,
        question,
        max_new_tokens,
    )
    if debug:
        print(
            "Inference input tokens:",
            inputs["input_ids"].shape[-1],
            "| context cropped:",
            was_cropped,
        )

    inference_model.eval()
    with torch.inference_mode():
        output_ids = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
            repetition_penalty=1.05,
            no_repeat_ngram_size=4,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )

    generated_ids = output_ids[0, inputs["input_ids"].shape[-1] :]
    if debug:
        stopped_at_eos = bool(
            (generated_ids == tokenizer.eos_token_id).any().item()
        )
        print(
            "Generated tokens:",
            len(generated_ids),
            "| stopped at EOS:",
            stopped_at_eos,
        )
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


def looks_like_token_collapse(text):
    token_ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    if len(token_ids) < 16:
        return False
    counts = Counter(token_ids)
    most_common_share = max(counts.values()) / len(token_ids)
    unique_share = len(counts) / len(token_ids)
    return most_common_share >= 0.35 or unique_share <= 0.15


def corrupted_characters(text):
    invalid_script_ranges = (
        (0x0B80, 0x0BFF),  # Tamil
        (0x0C00, 0x0C7F),  # Telugu
        (0x0C80, 0x0CFF),  # Kannada
        (0x0D00, 0x0D7F),  # Malayalam
    )
    bidi_controls = {0x200E, 0x200F, *range(0x202A, 0x202F), *range(0x2066, 0x206A)}
    return sorted(
        {
            (character, f"U+{ord(character):04X}")
            for character in text
            if character == "\ufffd"
            or ord(character) in bidi_controls
            or any(
                start <= ord(character) <= end
                for start, end in invalid_script_ranges
            )
        },
        key=lambda item: item[1],
    )


def looks_like_corrupted_sinhala(text):
    return not text.strip() or bool(corrupted_characters(text))


sanity_examples = [
    item for item in validation_rows if item["answerable"]
][:SANITY_EXAMPLES_PER_CLASS]
sanity_examples += [
    item for item in validation_rows if not item["answerable"]
][:SANITY_EXAMPLES_PER_CLASS]

sanity_failures = []
for example_index, example in enumerate(sanity_examples, 1):
    expected = example["answer"].strip() if example["answerable"] else NO_ANSWER
    predicted = ask_from_context(
        example["context"],
        example["question"],
        debug=True,
    )
    print("Question:", example["question"])
    print("Expected:", expected)
    print("Predicted:", predicted)
    print("Answerable:", example["answerable"])

    issues = []
    if looks_like_token_collapse(predicted):
        issues.append("repetition collapse")
    if looks_like_corrupted_sinhala(predicted):
        issues.append("empty, corrupted, or non-Sinhala-script output")

    if issues:
        bad_characters = corrupted_characters(predicted)
        if bad_characters:
            print("Flagged characters:", bad_characters)
        with inference_model.disable_adapter():
            base_prediction = ask_from_context(
                example["context"],
                example["question"],
            )
        print("Base model without LoRA:", base_prediction)
        print("Sanity issues:", ", ".join(issues))
        sanity_failures.append(
            {
                "example": example_index,
                "issues": issues,
                "prediction": predicted,
                "base_prediction": base_prediction,
            }
        )
    print("-" * 80)

SANITY_CHECKS_PASSED = not sanity_failures
print(
    "Generation sanity result:",
    "PASS" if SANITY_CHECKS_PASSED else f"FAIL ({len(sanity_failures)} examples)",
)
if not SANITY_CHECKS_PASSED:
    print("Do not run the export cell for this adapter.")

In [ ]:
# Export only an adapter that passed the generation sanity checks.
if not globals().get("SANITY_CHECKS_PASSED", False):
    raise RuntimeError(
        "Generation sanity checks did not pass. The adapter was not saved, merged, or published."
    )

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(ADAPTER_DIR)
print("Verified adapter saved to:", ADAPTER_DIR.resolve())

export_model = inference_model
if MERGE_MODEL:
    print("Adapter checks passed; merging into the base model...")
    export_model = trainer.model.merge_and_unload()
    export_model.config.use_cache = True
    MERGED_DIR.mkdir(parents=True, exist_ok=True)
    export_model.save_pretrained(
        MERGED_DIR,
        safe_serialization=True,
        max_shard_size="5GB",
    )
    tokenizer.save_pretrained(MERGED_DIR)
    print("Merged model saved to:", MERGED_DIR.resolve())
else:
    print("MERGE_MODEL is False; keeping the verified LoRA adapter unmerged.")

if PUSH_TO_HUB:
    if not os.environ.get("HF_TOKEN"):
        raise RuntimeError("Add HF_TOKEN to Modal Secrets before pushing.")
    export_model.push_to_hub(HF_REPO_ID, token=os.environ["HF_TOKEN"])
    tokenizer.push_to_hub(HF_REPO_ID, token=os.environ["HF_TOKEN"])
    print("Pushed to:", f"https://huggingface.co/{HF_REPO_ID}")
else:
    print("PUSH_TO_HUB is False; model remains in the notebook filesystem.")